In [37]:
import torch
import torch.nn.functional as F
import imageio
import numpy as np

In [38]:
import os
import torch
import sys
import matplotlib.pyplot as plt
from gsplat import rasterization
from optical.surface import WaterSurface
from optical.utils.test_utils import LoadDataset
from optical.utils.test_utils import combert_into_colormap, compare_rasterization_variants

%matplotlib inline
plt.rcParams['figure.figsize'] = (15.0, 12.0) # set default size of plots
plt.rcParams['image.interpolation'] = 'nearest'
plt.rcParams['image.cmap'] = 'gray'
# for auto-reloading extenrnal modules
%load_ext autoreload
%autoreload 2

device = torch.device("cuda:0")

Dataset = LoadDataset()
Dataset.set_iamge(80)

means, scales, quats, opacities, colors = Dataset.get_model_param()
pixels_wo, pixels_w, camtoworld, worldtocam, cam_center, K, height, width, image_id = Dataset.get_camera_param()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Loading 3D point cloud from: /home/taiki/dataset/river2/wo_refraction/points3d.ply
[Parser] Loaded 100 images from /home/taiki/dataset/river2/wo_refraction.
Loading 3D point cloud from: /home/taiki/dataset/river2/refraction/points3d.ply
[Parser] Loaded 100 images from /home/taiki/dataset/river2/refraction.


OutOfMemoryError: CUDA out of memory. Tried to allocate 20.00 MiB. GPU 0 has a total capacty of 11.75 GiB of which 54.75 MiB is free. Process 1873606 has 5.34 GiB memory in use. Process 2120774 has 780.00 MiB memory in use. Including non-PyTorch memory, this process has 4.98 GiB memory in use. Of the allocated memory 4.53 GiB is allocated by PyTorch, and 291.24 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting max_split_size_mb to avoid fragmentation.  See documentation for Memory Management and PYTORCH_CUDA_ALLOC_CONF

In [ ]:
WS = WaterSurface(
    means=means,
    scales=scales,
    quats=quats,
    opacities=opacities,
    device=device,
    
    camtoworld=camtoworld[0],
    K=K[0],
    width=width,
    height=height,
)

t_means, t_quats = WS.transform_to_appearance()
t_scales = WS.scale_correction_as_real()
map_colors = combert_into_colormap(WS.scale_correction_factor, colormap='plasma')

rays_WS = WS.calc_ray_of_each_pixel()
WS.load_environment_map(envmap_path="rainforest_trail_2k.exr")
env_colors = WS.get_colors_from_envmap(rays_WS)

WS.fresnel_reflectance()

In [ ]:
# refrected rays and get env map colors
reflected_rays = WS.calc_refrected_ray()
refrected_env_colors = WS.get_colors_from_envmap(reflected_rays)

refrected_env_colors_np = refrected_env_colors.cpu().numpy()
rgb = refrected_env_colors_np[:,:,:3]
a = refrected_env_colors_np[:,:,3].clip(0, 1) * 0.5

plt.figure(figsize=(10, 5))
plt.imshow(rgb, alpha=a)
plt.axis('off')
plt.title('Environment Colors from Water Surface')
plt.show()

In [ ]:
rgb, _, _ = rasterization(
    means=t_means,
    scales=t_scales,
    quats=t_quats,
    opacities=opacities,
    colors=colors,
    viewmats=worldtocam[0].unsqueeze(0),
    Ks=K[0].unsqueeze(0),
    width=width,
    height=height,
    sh_degree=0
)

img = rgb.squeeze().detach().cpu().numpy()

plt.figure(figsize=(10, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Rasterized Colors from Water Surface')
plt.show()


In [ ]:
incidence_angle = WS.incidence_angle
refraction_angle = WS.refraction_angle

spec_ratio, trans, spec = WS.fresnel_reflectance()

alpha = refrected_env_colors[:, :, 3].unsqueeze(-1) 
env_rgb = refrected_env_colors[:, :, :3] 

In [ ]:

reflected_img = trans * rgb.unsqueeze(0) \
    #+  spec * alpha * env_rgb 

In [ ]:
# show
plt.figure(figsize=(10, 5))
plt.imshow(reflected_img.squeeze().detach().cpu().numpy())
plt.axis('off')
plt.title('Reflected Image from Water Surface')
plt.show()
